# Export deployment assets for the web application

Run this in Colab **after** the final model notebook has produced `notebook2_outputs/`.
It retrains nothing: it reads the saved model and the saved test predictions, recovers the probeset
order from the series matrix, and packages everything the web application needs.

Before deploying anything it checks that

1. the series matrix is the same file the notebook used (SHA-256 from `run_manifest.json`), and
2. the exported model reproduces the test scores recorded in that run (to within 1e-5).

If either check fails the notebook stops with an error. Do not deploy assets from a failed run.

**Output:** `deploy_assets.zip`, which you unzip into the `assets/` folder of the application repository.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -q xgboost


## Export

In [ ]:
import os, io, gzip, json, hashlib, shutil
import numpy as np
import pandas as pd
import xgboost as xgb

# ---- paths: edit to match your Drive ----
MATRIX_PATH = "/content/drive/MyDrive/LSS_THESIS/dataset/GSE98320_series_matrix.txt.gz"
NB2_DIR     = "/content/drive/MyDrive/LSS_THESIS/notebook2_outputs"
OUT_DIR     = "/content/drive/MyDrive/LSS_THESIS/deploy_assets"

THESIS_TITLE = ("Predicting Kidney Graft Rejection from Biopsy Transcriptomic Data "
                "with a Bayesian-Optimized XGBoost Model")
AUTHORS      = ["Francis Joseph G. Libres", "Hanz Ian B. Silva", "Kim Miguel P. Sobrepeña"]
INSTITUTION  = "School of Information Technology · Mapúa University"
N_DEMO_PER_CLASS = 4          # demo biopsies bundled with the app, per class

os.makedirs(OUT_DIR, exist_ok=True)


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


# ---- 1. probeset order (must match Notebook 2 exactly: same parser, same AFFX removal) ----
def parse_geo_series_matrix(path):
    table_lines, in_table = [], False
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("!series_matrix_table_begin"):
                in_table = True; continue
            if line.startswith("!series_matrix_table_end"):
                break
            if in_table:
                table_lines.append(line)
    expr = pd.read_csv(io.StringIO("\n".join(table_lines)), sep="\t", index_col=0).astype(np.float32)
    expr.index = expr.index.astype(str)
    return expr

expr_df = parse_geo_series_matrix(MATRIX_PATH)
expr_df = expr_df.loc[~expr_df.index.str.startswith("AFFX")]
PROBES = expr_df.index.to_list()
print(f"Probesets: {len(PROBES):,} | samples in matrix: {expr_df.shape[1]}")

with open(f"{NB2_DIR}/run_manifest.json") as fh:
    MANIFEST = json.load(fh)
assert MANIFEST["sha256"]["series_matrix"] == sha256(MATRIX_PATH), \
    "This series matrix is not the file Notebook 2 used — the probeset order would not match the model."
assert len(PROBES) == MANIFEST["data"]["n_probesets"], "Probeset count differs from the notebook run."

# ---- 2. model + saved test predictions ----
MODEL_SRC = f"{NB2_DIR}/final_tuned_xgboost.json"
model = xgb.XGBClassifier()
model.load_model(MODEL_SRC)
n_trees = len(model.get_booster().get_dump())

pred = pd.read_csv(f"{NB2_DIR}/test_predictions.csv")
prob_col = "prob_tuned_xgboost"
assert prob_col in pred.columns, f"{prob_col} missing from test_predictions.csv"

# ---- 3. demo biopsies: a spread of scores in each class, taken from the held-out test set ----
chosen = []
for lab in (1, 0):
    sub = pred[pred.label == lab].sort_values(prob_col).reset_index(drop=True)
    picks = np.unique(np.linspace(0, len(sub) - 1, N_DEMO_PER_CLASS).round().astype(int))
    chosen.append(sub.loc[picks])
demo = pd.concat(chosen).sort_values(prob_col, ascending=False).reset_index(drop=True)

X_demo = expr_df.loc[PROBES, demo.gsm.to_list()].to_numpy(dtype=np.float32).T
probs_now = model.predict_proba(X_demo)[:, 1]
diff = np.max(np.abs(probs_now - demo[prob_col].to_numpy()))
print(f"Demo biopsies: {len(demo)} | max |score difference| vs the notebook run: {diff:.2e}")
assert diff < 1e-5, "The exported model does not reproduce the notebook's test scores — do not deploy."

np.savez_compressed(f"{OUT_DIR}/demo_specimens.npz",
                    X=X_demo, gsm=demo.gsm.to_numpy().astype(object),
                    label=demo.label.to_numpy().astype(int),
                    recorded_prob=demo[prob_col].to_numpy().astype(float))

# ---- 4. probeset list, data dictionary, model file ----
with gzip.open(f"{OUT_DIR}/probesets.txt.gz", "wt") as fh:
    fh.write("\n".join(PROBES))
shutil.copy2(MODEL_SRC, f"{OUT_DIR}/final_tuned_xgboost.json")

dic = pd.read_csv(f"{NB2_DIR}/data_dictionary_top20.csv")
dic = dic[["rank", "probeset", "gene_symbol", "gain", "mean_log2_rejection",
           "mean_log2_non_rejection", "log2_fold_change", "higher_in"]]
dic.to_csv(f"{OUT_DIR}/top20_dictionary.csv", index=False)

# ---- 5. metadata for the About page ----
test_point = pd.read_csv(f"{NB2_DIR}/test_metrics_point.csv", index_col=0)
cv_folds = pd.read_csv(f"{NB2_DIR}/cv_fold_results.csv")
cv_tuned = cv_folds[cv_folds.model == "Tuned XGBoost"].mean(numeric_only=True)
METRICS = ["roc_auc", "auprc", "f1", "sensitivity", "specificity", "precision", "accuracy"]

meta = {
    "thesis_title": THESIS_TITLE, "authors": AUTHORS, "institution": INSTITUTION,
    "dataset": "GSE98320", "platform": "Affymetrix PrimeView (GPL15207)", "threshold": 0.5,
    "n_probesets": len(PROBES),
    "n_train": MANIFEST["data"]["n_train"], "n_test": MANIFEST["data"]["n_test"],
    "n_train_rejection": MANIFEST["data"]["n_train_rejection"],
    "n_test_rejection": MANIFEST["data"]["n_test_rejection"],
    "n_probesets_used": MANIFEST["final_model"]["n_probesets_used"], "n_trees": n_trees,
    "best_params": MANIFEST["optuna"]["best_params"],
    "best_cv_auc": MANIFEST["optuna"]["best_cv_auc"], "n_trials": MANIFEST["optuna"]["n_trials"],
    "test_metrics": {m: round(float(test_point.loc["Tuned XGBoost", m]), 4) for m in METRICS},
    "cv_metrics_tuned": {m: round(float(cv_tuned[m]), 4) for m in METRICS},
    "versions": MANIFEST["versions"], "run_finished_utc": MANIFEST.get("run_finished_utc", ""),
    "series_matrix_sha256": MANIFEST["sha256"]["series_matrix"],
    "model_sha256": sha256(MODEL_SRC),
    "demo_specimens": demo.gsm.to_list(),
}
with open(f"{OUT_DIR}/deploy_meta.json", "w") as fh:
    json.dump(meta, fh, indent=2)

# ---- 6. zip for download ----
zip_path = shutil.make_archive("/content/deploy_assets", "zip", OUT_DIR)
print("\nExported to", OUT_DIR)
for fn in sorted(os.listdir(OUT_DIR)):
    print(f"   {fn:<30} {os.path.getsize(f'{OUT_DIR}/{fn}') / 1e6:7.2f} MB")
print(f"\nZip for the repository: {zip_path} ({os.path.getsize(zip_path) / 1e6:.2f} MB)")
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Automatic download unavailable:", e, "— download the zip from the Colab file browser.")
